## Feature Engineering

**Fuel lag features**

Airlines don't react to this month's fuel price. They react to trends over the past quarter. Add fuel prices from the past three months:

In [19]:
import pandas as pd

df = pd.read_csv('../data/processed/model_ready_no_covid.csv')
df = df.sort_values(['country', 'year_month']).reset_index(drop=True)

df['fuel_lag1'] = df.groupby('country')['jet_fuel_usd_per_gallon'].shift(1)
df['fuel_lag2'] = df.groupby('country')['jet_fuel_usd_per_gallon'].shift(2)
df['fuel_lag3'] = df.groupby('country')['jet_fuel_usd_per_gallon'].shift(3)

**Fuel change features**

A sudden spike is more disruptive than a gradual rise.

In [20]:
df['fuel_change_1m'] = df['jet_fuel_usd_per_gallon'].pct_change(1) * 100
df['fuel_change_3m'] = df['jet_fuel_usd_per_gallon'].pct_change(3) * 100
df['fuel_change_6m'] = df['jet_fuel_usd_per_gallon'].pct_change(6) * 100
df['fuel_volatility_6m'] = df['jet_fuel_usd_per_gallon'].rolling(6).std()

**Seasonal features**

Month of year matters significantly for air travel.

In [21]:
df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month'] = df['year_month_dt'].dt.month
df['quarter'] = df['year_month_dt'].dt.quarter

# Peak travel months (June-August, December)
df['is_peak_season'] = df['month'].isin([6, 7, 8, 12]).astype(int)

**GDP change feature**

To determine whether a country's economy is growing or shrinking.

In [22]:
df['gdp_yoy_change'] = df.groupby('country')['gdp_per_capita'].pct_change(1) * 100

**Encoding the LCC flag**

In [24]:
import pandas as pd

routes = pd.read_csv('../data/processed/routes.csv')
airline_types = pd.read_csv('../data/processed/airline_classifications_clean.csv')

routes_enriched = routes.merge(airline_types, on='airline_iata', how='left')

country_features = routes_enriched.groupby('destination_country').agg(
    total_routes=('airline_iata', 'count'),
    lcc_routes=('carrier_type', lambda x: (x == 'LCC').sum())
).reset_index()

country_features['lcc_share'] = country_features['lcc_routes'] / country_features['total_routes']

df = df.merge(country_features, left_on='country', right_on='destination_country', how='left')

# now this will work
df['is_high_lcc_corridor'] = (df['lcc_share'] > 0.4).astype(int)

In [25]:
df['is_high_lcc_corridor'] = (df['lcc_share'] > 0.4).astype(int)

**Final feature set**